<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing L2 data (CALLIPSO)**

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> Run Notebook in binder from NASA-tutorial Github repository, or
3. <font size="3"> Get `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
- <font size="3"> Inspect metadata.
- <font size="3"> Subset by variable names, time and DayTime only
- <font size="3"> Stream data into local files


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

<font size="3"> We query NASA's CMR to identify remote files that intersect the following geographical area (bounding box) covering the following time range

- <font size="3"> -121 < longitude < -115, and 26.5 < latitude < 31
- <font size="3"> 2 years of only spring time data: March 1st to May 31st (2022-2023).

 <font size="3"> Lasly, we are **ONLY** interested in `Daytime` data.



In [ ]:
Callipso_L2_ccid = "C3463063995-LARC_CLOUD" # 
bbox = [-121,26.5,-115,31] # [west, south, east, north]

# 10 years of March data
time_ranges = [[dt.datetime(year, 3, 1), dt.datetime(year, 5, 31)] for year in range(2022, 2024)]

CMR_URLs = []
args = {
    "ccid": Callipso_L2_ccid,
    "bounding_box": bbox,
    "limit": 1000,
}
cmr_urls = [url for time_range in time_ranges for url in get_cmr_urls(**args, time_range=time_range)] # you can increase the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


### Accessing Metadata-ONLY with PyDAP

In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
output_path = "data/"

## Download minimal variables to identify spatial subset and daytime data

<font size="3"> Coordinates are have (fully qualifying names):

- <font size="3"> `Latitude`
- <font size="3"> `Longitude`
- <font size="3"> `Day_Night_Flag`


<font size="3"> Before donwloading, we need to idenfity any dimension that is also an array of the dataset

<font size="3"> (There can also be Named dimensions, i.e. dimensions that are only named and that are NOT

<font size="3">associated with any data. We do not need to declare those Variables)


In [ ]:
DIMS = list(set(pyds['Latitude'].dims + pyds['Longitude'].dims + pyds['Day_Night_Flag'].dims))
dims = [dim for dim in DIMS if dim.split("/")[-1] in pyds[("/").join(DIMS[1].split('/')[:-1])].variables()] 
print("Dimensions that are also arrays: ", dims)

### Stream data

In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables = ["/Longitude", "/Day_Night_Flag"], 
              output_path=output_path)

## Inspect all downloaded the data

<font size="3"> And identify the SLICE per granule


In [ ]:
# Get data from Bounding Box
minLon, maxLon = bbox[0], bbox[2]

slices=[]
final_urls = []
for url in cmr_urls:
    filename = output_path+f"{url.split('/')[-1][:-4]}.nc4"
    dt1 = xr.open_datatree(filename).load()
    daytime_flag = dt1['Day_Night_Flag']
    # find index /data_01/longitude
    longitude = dt1['/Longitude']
    mask = (longitude >= minLon) & (longitude <= maxLon)
    idx = np.nonzero(mask.values)[0]
    daytime_flag = dt1['Day_Night_Flag'].isel(Record_Number=slice(idx[0], idx[-1]))==1
    if all(daytime_flag==0):
        final_urls.append(url)
        slices.append({"/Record_Number":(idx[0], idx[-1])})

print(f"\nOnly {len(final_urls)} out of the {len(cmr_urls)} remote files satisfy our Daylight Criteria\n")
print("Sample subsetting slices:")
slices[:4]

## Inspect Visually the slice to subset

In [ ]:
# Subset data
Lon = dt1['Longitude'].isel(Record_Number=slice(idx[0], idx[-1]))

# Generate masked data to visualize only

Lon_masked = xr.full_like(dt1['Longitude'], np.nan)
Lon_masked.loc[dict(
    Record_Number = Lon['Record_Number'] + idx[0]
)] = Lon

# Visualize: Plot subset of data over original data
fig, axes = plt.subplots(figsize=(10,4))
dt1['Longitude'].plot(lw=5, color='k', alpha=0.75);
Lon_masked.plot(lw=10, color="#7f00ff")
axes.set_title(r"Longitude Subset $[^\circ$E]")
plt.show()

## Download all data of interest

<font size="3"> **NOTE:** Erase all previously downloaded files, to avoid filename collision


In [ ]:
# Will Download 34 Variables!
keep_variables = [
    '/Lidar_Surface_Detection', # <----- ALL Variables inside Group
    "/Ocean_Derived_Column_Optical_Depth", # < -- All varibles inside Group
    "/Lidar_Data_Altitudes", "/Profile_ID", "/Latitude", "/Longitude", 
    "/Profile_Time", "/Profile_UTC_Time", "/Day_Night_Flag", "/Tropopause_Height", 
    "/Tropopause_Temperature",
]


In [ ]:
%%time
dap_to_netcdf(final_urls, session=my_session, 
              keep_variables = keep_variables, 
              dim_slices = slices,
              output_path=output_path)

## Inspect a downloaded (local) file

<font size="3"> **NOTE:** File inherits the source filename via the OPeNDAP metadata. We can retrieve the source filename from each URL


In [ ]:
filename = output_path+f"{final_urls[0].split('/')[-1][:-4]}.nc4"
dt1 = xr.open_datatree(filename).load()
dt1
